# 05 — Ransomware leak-site ingestion (basic)

**Goal.** Pull recent ransomware victim postings from the public ransomware.live API and persist them into the same `documents` table you built in notebook 03. From there, notebook 04's classifier and notebook 06's monitoring loop work over leak-site and onion content uniformly.

**Why this source.** ransomware.live continuously scrapes the public leak sites of ~80 ransomware groups (LockBit, ALPHV/BlackCat, Cl0p, RansomHub, Akira, …) and exposes a free, no-auth REST API. For a starter monitoring stack it's the closest you can get to "what's leaking right now" without API keys, paid feeds, or Telegram credentials.

**Why we replaced Telegram.** The original plan was Telegram-channel ingestion, but the Telegram API requires credentials from `my.telegram.org/apps`, which is unreliable to provision. ransomware.live gives us the same monitoring signal — fresh leak-site activity — with a one-line `requests.get`.

## Read this before running

- **Public data only.** ransomware.live publishes what the leak sites themselves publish. Treat it as read-only intelligence; do not redistribute victim PII.
- **No credentials needed.** The free `/v2/recentvictims` endpoint requires no key.
- **Rate limits.** The API is generous but not unlimited. We fetch once per run; don't loop on it.

## 1. Setup

Nothing extra to install — `requests` and `duckdb` are already in your env from earlier notebooks.

The endpoint we use is `https://api.ransomware.live/v2/recentvictims`. It returns the ~100 most recent victim postings as a JSON array.

In [37]:
import os, json, hashlib
from pathlib import Path
from datetime import datetime, timezone, timedelta
import requests
import duckdb

API_URL = "https://api.ransomware.live/v2/recentvictims"
USER_AGENT = "Mozilla/5.0 (cti_401 research collector; contact: you@example.com)"

RW_DIR = Path("data/ransomware")
RW_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = Path("db/cti.duckdb")

## 2. Filters

The API returns the ~100 most recent victims regardless of age. We add two cheap filters:

- `LOOKBACK_DAYS` — drop anything posted before this window. Leak sites sometimes re-publish old victims, so this prevents stale rows from polluting the corpus on each run.
- `MIN_CHARS` — skip records where the synthesized document is shorter than this. A "victim: ACME / group: lockbit" line with no description isn't worth classifying.

In [38]:
LOOKBACK_DAYS = 5000   # only keep victims posted within this window
MIN_CHARS = 80       # skip near-empty records

print(f"keeping victims posted in the last {LOOKBACK_DAYS} days, "
      f"with synthesized text >= {MIN_CHARS} chars")

keeping victims posted in the last 5000 days, with synthesized text >= 80 chars


## 3. Pull recent victims

We persist the raw API response to JSONL first (cheap, append-only) and then load into DuckDB at the end. If the API is flaky mid-run we still have a usable snapshot.

Each victim record has fields like `victim`, `group`, `activity` (sector), `country`, `attackdate`, `discovered`, `description`, `url`. We synthesize a single text blob from the meaningful fields so the downstream classifier (nb 04) has something to chew on.

In [39]:
def parse_dt(s):
    """ransomware.live dates look like '2026-04-15T00:00:00+00:00' or ''."""
    if not s:
        return None
    try:
        return datetime.fromisoformat(s.replace("Z", "+00:00"))
    except ValueError:
        return None

def synth_text(v):
    """Build a classifiable text blob from a victim record."""
    parts = []
    if v.get("victim"):     parts.append(f"Victim: {v['victim']}")
    if v.get("group"):      parts.append(f"Ransomware group: {v['group']}")
    if v.get("activity"):   parts.append(f"Sector: {v['activity']}")
    if v.get("country"):    parts.append(f"Country: {v['country']}")
    if v.get("domain"):     parts.append(f"Domain: {v['domain']}")
    if v.get("attackdate"): parts.append(f"Attack date: {v['attackdate'][:10]}")
    if v.get("description"):parts.append(v["description"].strip())
    return "\n".join(parts)

# 1. Fetch
r = requests.get(API_URL, headers={"User-Agent": USER_AGENT}, timeout=30)
r.raise_for_status()
victims = r.json()
print(f"fetched {len(victims)} recent victims from ransomware.live")

# 2. Snapshot to JSONL (one file per run)
snapshot_path = RW_DIR / f"victims_{datetime.now(timezone.utc):%Y%m%dT%H%M%S}.jsonl"
with snapshot_path.open("w") as f:
    for v in victims:
        f.write(json.dumps(v, ensure_ascii=False) + "\n")
print(f"snapshot -> {snapshot_path}")

# 3. Filter
since = datetime.now(timezone.utc) - timedelta(days=LOOKBACK_DAYS)
kept = []
for v in victims:
    dt = parse_dt(v.get("discovered")) or parse_dt(v.get("attackdate"))
    if dt is None or dt < since:
        continue
    text = synth_text(v)
    if len(text) < MIN_CHARS:
        continue
    kept.append((v, dt, text))

print(f"{len(kept)} victims pass the {LOOKBACK_DAYS}-day / {MIN_CHARS}-char filters")

Task was destroyed but it is pending!
task: <Task pending name='Task-405' coro=<_async_in_context.<locals>.run_in_context() done, defined at /home/saqib/venv/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-406' coro=<Kernel.shell_main() running at /home/saqib/venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /home/saqib/venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:563]>
/usr/lib/python3.12/collections/__init__.py:447: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  @classmethod
Task was destroyed but it is pending!
task: <Task pending name='Task-406' coro=<Kernel.shell_main() running at /home/saqib/venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]>


fetched 100 recent victims from ransomware.live
snapshot -> data/ransomware/victims_20260429T154924.jsonl
100 victims pass the 5000-day / 80-char filters


## 4. Normalize into `documents`

Schema match with nb 03: each victim becomes one row in the same `documents` table.

- `url` is the ransomware.live victim page (`https://www.ransomware.live/id/...`) — stable across re-runs, so `url_id = sha1(url)` makes the primary key idempotent.
- `source = 'ransomware_live'` distinguishes these rows from `'onion'` (nb 03).
- Each victim is its own `dedup_group`; cross-victim deduplication is not interesting here (different companies, different dates).

In [40]:
def url_id(url): return hashlib.sha1(url.encode()).hexdigest()

rows = []
for v, dt, text in kept:
    url = v.get("url") or f"ransomware.live://{v.get('victim','')}@{v.get('group','')}"
    title = f"{v.get('group','?')} -> {v.get('victim','?')}"
    rows.append((
        url_id(url), url, "ransomware_live",
        title, text, len(text), dt.isoformat(),
        None,         # dedup_group (set below)
        None, None,   # label, score (nb 04 fills these)
    ))

if not rows:
    print("No rows to insert. Widen LOOKBACK_DAYS in section 2 and re-run section 3.")
else:
    con = duckdb.connect(str(DB_PATH))
    con.executemany("""
        INSERT OR REPLACE INTO documents
        (url_id, url, source, title, text, n_chars, fetched_at, dedup_group, label, score)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, rows)

    # Assign each new ransomware_live row its own dedup_group. DuckDB disallows
    # window functions inside UPDATE SET, so we compute row numbers in a CTE
    # and join back.
    max_grp = con.execute("SELECT COALESCE(MAX(dedup_group), -1) FROM documents").fetchone()[0]
    con.execute(f"""
        WITH new_groups AS (
            SELECT url_id,
                   {max_grp + 1} + ROW_NUMBER() OVER (ORDER BY url_id) AS new_group
            FROM documents
            WHERE source = 'ransomware_live' AND dedup_group IS NULL
        )
        UPDATE documents AS d
        SET dedup_group = ng.new_group
        FROM new_groups AS ng
        WHERE d.url_id = ng.url_id
    """)

    print(con.execute("SELECT source, COUNT(*) FROM documents GROUP BY source").fetchdf())
    con.close()

            source  count_star()
0  ransomware_live           100
1            onion             4


## 5. Re-run notebook 04 to classify the new rows

Notebook 04's update query only touches rows where `label IS NULL`, so re-running it is cheap and only processes the new ransomware.live records.

> Reminder: shut down nb 04's kernel (or `con.close()` there) before re-running, or you'll hit a DuckDB lock conflict — only one writer at a time.

## What's next

Notebook 06 stitches everything together: a periodic re-fetch (onion + ransomware.live), classification of new content, and an alert feed for high-severity hits. Update nb 06's severity policy if needed — many `ransomware_live` rows will land under whatever your DarkBERT model calls "Hacking" / "Cybercrime".